# Notebook 01: Data Ingestion & Cleaning

**Goal:** Load the raw anime dataset, explore its structure, fix data types, handle missing values, and save a clean version.

**Steps:**
1. Import libraries
2. Load raw CSV
3. Inspect structure and data types
4. Parse JSON fields (characters)
5. Normalize multi-value fields
6. Handle missing values
7. Save cleaned data

---

## 1. Import Libraries & Setup

In [1]:
# Core libraries
import pandas as pd
import numpy as np
import json
import ast
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print("✅ Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

✅ Libraries imported successfully
Pandas version: 2.3.3
NumPy version: 1.26.4


## 2. Load Raw Dataset

Loading `mal_anime.csv` and performing initial inspection.

In [2]:
# Define paths
DATA_DIR = Path('data')
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'

# Create directories if they don't exist
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Load the dataset
df = pd.read_csv('mal_anime.csv')

print(f"✅ Dataset loaded successfully")
print(f"Shape: {df.shape}")
print(f"\nColumn names and types:")
print(df.dtypes)
print(f"\n{'='*60}")
print(f"First few rows:")
print(df.head(3))

✅ Dataset loaded successfully
Shape: (19931, 25)

Column names and types:
myanimelist_id       int64
title               object
description         object
image               object
Type                object
Episodes            object
Status              object
Premiered           object
Released_Season     object
Released_Year      float64
Source              object
Genres              object
Themes              object
Studios             object
Producers           object
Demographic         object
Duration            object
Rating              object
Score              float64
Ranked              object
Popularity          object
Members             object
Favorites           object
characters          object
source_url          object
dtype: object

First few rows:
   myanimelist_id                            title  \
0               1                     Cowboy Bebop   
1               5  Cowboy Bebop: Tengoku no Tobira   
2               6                           Trigun   

   

## 3. Data Quality Assessment

Inspect missing values, data types, and identify fields requiring transformation.

In [3]:
# Missing value analysis
missing_summary = pd.DataFrame({
    'column': df.columns,
    'missing_count': df.isnull().sum(),
    'missing_percent': (df.isnull().sum() / len(df) * 100).round(2),
    'dtype': df.dtypes
})

missing_summary = missing_summary[missing_summary['missing_count'] > 0].sort_values(
    'missing_count', ascending=False
)

print("Missing Values Summary:")
print(missing_summary.to_string(index=False))

print(f"\n{'='*60}")
print("\nSample values for key columns:")
print(f"\nEpisodes (first 5): {df['Episodes'].head().tolist()}")
print(f"\nRanked (first 5): {df['Ranked'].head().tolist()}")
print(f"\nPopularity (first 5): {df['Popularity'].head().tolist()}")
print(f"\nMembers (first 5): {df['Members'].head().tolist()}")
print(f"\nFavorites (first 5): {df['Favorites'].head().tolist()}")

Missing Values Summary:
         column  missing_count  missing_percent   dtype
Released_Season          13749            68.98  object
  Released_Year          13749            68.98 float64
      Premiered          13588            68.18  object
    Demographic          13409            67.28  object
         Themes           8714            43.72  object
     characters           6412            32.17  object
          Score           4692            23.54 float64
         Ranked           2525            12.67  object
         Genres           1661             8.33  object
         Rating            874             4.39  object
         Source            765             3.84  object
           Type            397             1.99  object
          image            387             1.94  object
      Favorites            382             1.92  object
        Members            382             1.92  object
     Popularity            382             1.92  object
        Studios         

## 4. Inspect Complex Fields

Examine JSON and list-like fields (characters, Genres, Themes, Studios, Producers).

In [4]:
# Inspect characters field (JSON)
print("Characters field (first non-null):")
first_chars = df[df['characters'].notna()]['characters'].iloc[0]
print(first_chars[:500])
print(f"\nType: {type(first_chars)}")

print(f"\n{'='*60}")

# Inspect multi-value text fields
print("\nMulti-value fields structure:")
for col in ['Genres', 'Themes', 'Studios', 'Producers']:
    sample = df[df[col].notna()][col].iloc[0]
    print(f"\n{col}:")
    print(f"  Sample: {sample}")
    print(f"  Type: {type(sample)}")
    
print(f"\n{'='*60}")

# Check unique values for categorical fields
print("\nUnique value counts:")
for col in ['Type', 'Status', 'Source', 'Rating']:
    n_unique = df[col].nunique()
    print(f"{col}: {n_unique} unique values")
    if n_unique < 15:
        print(f"  Values: {df[col].value_counts().to_dict()}")

Characters field (first non-null):
[{"id": 3, "name": "Black, Jet", "url": "https://myanimelist.net/character/3/Jet_Black"}, {"id": 1, "name": "Spiegel, Spike", "url": "https://myanimelist.net/character/1/Spike_Spiegel"}, {"id": 2, "name": "Valentine, Faye", "url": "https://myanimelist.net/character/2/Faye_Valentine"}, {"id": 16, "name": "Wong Hau Pepelu Tivrusky IV, Edward", "url": "https://myanimelist.net/character/16/Edward_Wong_Hau_Pepelu_Tivrusky_IV"}, {"id": 131427, "name": "Actress", "url": "https://myanimelist.net/charact

Type: <class 'str'>


Multi-value fields structure:

Genres:
  Sample: Action, Award Winning, Sci-Fi
  Type: <class 'str'>

Themes:
  Sample: Adult Cast, Space
  Type: <class 'str'>

Studios:
  Sample: Sunrise
  Type: <class 'str'>

Producers:
  Sample: Bandai Visual, Victor Entertainment, Audio Planning U
  Type: <class 'str'>


Unique value counts:
Type: 7 unique values
  Values: {'TV': 6343, 'ONA': 3733, 'OVA': 3681, 'Movie': 3382, 'Special': 1587, 'TV Spe

## 5. Data Type Conversion

Convert string-encoded numeric fields to proper numeric types.

In [5]:
df_clean = df.copy()

# Convert Episodes to numeric
df_clean['Episodes'] = pd.to_numeric(df_clean['Episodes'], errors='coerce')

# Clean and convert Ranked (remove '#' prefix)
df_clean['Ranked'] = df_clean['Ranked'].str.replace('#', '', regex=False)
df_clean['Ranked'] = pd.to_numeric(df_clean['Ranked'], errors='coerce')

# Clean and convert Popularity (remove '#' prefix)
df_clean['Popularity'] = df_clean['Popularity'].str.replace('#', '', regex=False)
df_clean['Popularity'] = pd.to_numeric(df_clean['Popularity'], errors='coerce')

# Clean and convert Members (remove commas)
df_clean['Members'] = df_clean['Members'].str.replace(',', '', regex=False)
df_clean['Members'] = pd.to_numeric(df_clean['Members'], errors='coerce')

# Clean and convert Favorites (remove commas)
df_clean['Favorites'] = df_clean['Favorites'].str.replace(',', '', regex=False)
df_clean['Favorites'] = pd.to_numeric(df_clean['Favorites'], errors='coerce')

# Convert Released_Year to int (will keep as float for now due to NaN)
df_clean['Released_Year'] = df_clean['Released_Year'].astype('Int64')

print("Data type conversions completed")
print("\nUpdated dtypes:")
print(df_clean[['Episodes', 'Ranked', 'Popularity', 'Members', 'Favorites', 'Released_Year']].dtypes)
print("\nSample converted values:")
print(df_clean[['Episodes', 'Ranked', 'Popularity', 'Members', 'Favorites']].head())

Data type conversions completed

Updated dtypes:
Episodes         float64
Ranked           float64
Popularity       float64
Members          float64
Favorites        float64
Released_Year      Int64
dtype: object

Sample converted values:
   Episodes  Ranked  Popularity    Members  Favorites
0      26.0    48.0        42.0  2008019.0    87916.0
1       1.0   232.0       649.0   403604.0     1748.0
2      26.0   385.0       265.0   815140.0    17193.0
3      26.0  3344.0      1979.0   125868.0      686.0
4      52.0  4887.0      5765.0    16456.0       18.0


## 6. Parse Characters JSON Field

Extract character names from JSON string and create a list of character names.

In [6]:
def parse_characters(char_string):
    """
    Parse characters JSON string and extract character names.
    Returns list of character names or empty list if parsing fails.
    """
    if pd.isna(char_string):
        return []
    
    try:
        char_list = json.loads(char_string)
        names = [char.get('name', '') for char in char_list if isinstance(char, dict)]
        return names
    except:
        return []

# Apply parsing
df_clean['character_list'] = df_clean['characters'].apply(parse_characters)

# Create character count feature
df_clean['character_count'] = df_clean['character_list'].apply(len)

# Create comma-separated character names for easier viewing
df_clean['character_names'] = df_clean['character_list'].apply(
    lambda x: ', '.join(x) if x else None
)

print("Characters parsed successfully")
print(f"\nCharacter statistics:")
print(f"Rows with characters: {(df_clean['character_count'] > 0).sum()}")
print(f"Mean characters per anime: {df_clean['character_count'].mean():.2f}")
print(f"Max characters: {df_clean['character_count'].max()}")

print("\nSample parsed characters:")
print(df_clean[['title', 'character_count', 'character_names']].head(3).to_string())

Characters parsed successfully

Character statistics:
Rows with characters: 13519
Mean characters per anime: 11.11
Max characters: 1931

Sample parsed characters:
                             title  character_count                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

## 7. Process Multi-Value Fields

Split comma-separated fields into lists and create cleaned versions.

In [7]:
def parse_comma_separated(value):
    """
    Parse comma-separated string into list of cleaned values.
    Returns empty list if value is NaN.
    """
    if pd.isna(value):
        return []
    return [item.strip() for item in str(value).split(',')]

# Process multi-value fields
multi_value_cols = ['Genres', 'Themes', 'Studios', 'Producers']

for col in multi_value_cols:
    list_col = f"{col.lower()}_list"
    df_clean[list_col] = df_clean[col].apply(parse_comma_separated)
    
    count_col = f"{col.lower()}_count"
    df_clean[count_col] = df_clean[list_col].apply(len)

print("Multi-value fields processed")

for col in multi_value_cols:
    list_col = f"{col.lower()}_list"
    count_col = f"{col.lower()}_count"
    non_empty = (df_clean[count_col] > 0).sum()
    mean_count = df_clean[count_col].mean()
    print(f"\n{col}:")
    print(f"  Non-empty rows: {non_empty}")
    print(f"  Mean count: {mean_count:.2f}")
    print(f"  Sample: {df_clean[list_col].iloc[0]}")

Multi-value fields processed

Genres:
  Non-empty rows: 18270
  Mean count: 1.81
  Sample: ['Action', 'Award Winning', 'Sci-Fi']

Themes:
  Non-empty rows: 11217
  Mean count: 0.88
  Sample: ['Adult Cast', 'Space']

Studios:
  Non-empty rows: 19549
  Mean count: 1.05
  Sample: ['Sunrise']

Producers:
  Non-empty rows: 19549
  Mean count: 1.85
  Sample: ['Bandai Visual', 'Victor Entertainment', 'Audio Planning U']


## 8. Handle Missing Values

Fill or flag missing values based on column semantics.

In [8]:
# Numeric fields: fill with 0 or median as appropriate
df_clean['Episodes'] = df_clean['Episodes'].fillna(1)
df_clean['Members'] = df_clean['Members'].fillna(0)
df_clean['Favorites'] = df_clean['Favorites'].fillna(0)

# Ranked and Popularity: keep NaN (unranked items)
# Score: keep NaN (not yet scored)

# Categorical fields: fill with 'Unknown'
categorical_cols = ['Type', 'Status', 'Source', 'Rating', 'Demographic']
for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna('Unknown')

# Text fields: fill with empty string
df_clean['description'] = df_clean['description'].fillna('')
df_clean['image'] = df_clean['image'].fillna('')

# Duration: fill with 'Unknown'
df_clean['Duration'] = df_clean['Duration'].fillna('Unknown')

# Premiered, Released_Season: keep as is (temporal info, NaN is informative)

print("Missing values handled")
print("\nRemaining missing values:")
missing_after = df_clean.isnull().sum()
missing_after = missing_after[missing_after > 0].sort_values(ascending=False)
print(missing_after)

print(f"\n{'='*60}")
print("\nDataset summary:")
print(f"Total rows: {len(df_clean)}")
print(f"Total columns: {len(df_clean.columns)}")
print(f"\nNew columns added:")
new_cols = [col for col in df_clean.columns if col not in df.columns]
print(new_cols)

Missing values handled

Remaining missing values:
Released_Season    13749
Released_Year      13749
Premiered          13588
Themes              8714
characters          6412
character_names     6412
Score               4692
Ranked              2525
Genres              1661
Studios              382
Producers            382
Popularity           382
dtype: int64


Dataset summary:
Total rows: 19931
Total columns: 36

New columns added:
['character_list', 'character_count', 'character_names', 'genres_list', 'genres_count', 'themes_list', 'themes_count', 'studios_list', 'studios_count', 'producers_list', 'producers_count']


## 9. Create Derived Features

Add useful derived columns for downstream modeling.

In [11]:
# Binary flags
df_clean['has_score'] = df_clean['Score'].notna().astype(int)
df_clean['has_rank'] = df_clean['Ranked'].notna().astype(int)
df_clean['is_completed'] = (df_clean['Status'] == 'Finished Airing').astype(int)

# Log-transformed popularity metrics
df_clean['log_members'] = np.log1p(df_clean['Members'])
df_clean['log_favorites'] = np.log1p(df_clean['Favorites'])

# Favorite rate
df_clean['favorite_rate'] = df_clean['Favorites'] / (df_clean['Members'] + 1)

# Primary genre and studio
df_clean['primary_genre'] = df_clean['genres_list'].apply(lambda x: x[0] if len(x) > 0 else 'Unknown')
df_clean['primary_studio'] = df_clean['studios_list'].apply(lambda x: x[0] if len(x) > 0 else 'Unknown')

# Year-based features (handle NaN in Int64 type)
df_clean['is_modern'] = df_clean['Released_Year'].apply(lambda x: 1 if pd.notna(x) and x >= 2010 else 0)
df_clean['decade'] = df_clean['Released_Year'].apply(lambda x: (x // 10 * 10) if pd.notna(x) else pd.NA).astype('Int64')

print("Derived features created")
print("\nNew feature summary:")
derived_features = [
    'has_score', 'has_rank', 'is_completed', 
    'log_members', 'log_favorites', 'favorite_rate',
    'primary_genre', 'primary_studio', 'is_modern', 'decade'
]

for feat in derived_features:
    if df_clean[feat].dtype in ['int64', 'float64', 'Int64']:
        print(f"\n{feat}: mean={df_clean[feat].mean():.3f}, std={df_clean[feat].std():.3f}")
    else:
        print(f"\n{feat}: {df_clean[feat].nunique()} unique values")
        print(f"  Top 3: {df_clean[feat].value_counts().head(3).to_dict()}")

Derived features created

New feature summary:

has_score: 2 unique values
  Top 3: {1: 15239, 0: 4692}

has_rank: 2 unique values
  Top 3: {1: 17406, 0: 2525}

is_completed: 2 unique values
  Top 3: {1: 18668, 0: 1263}

log_members: mean=8.230, std=2.586

log_favorites: mean=2.401, std=2.465

favorite_rate: mean=0.003, std=0.004

primary_genre: 22 unique values
  Top 3: {'Comedy': 4961, 'Action': 4291, 'Adventure': 2201}

primary_studio: 1058 unique values
  Top 3: {'add some': 4391, 'Toei Animation': 824, 'Sunrise': 535}

is_modern: mean=0.194, std=0.395

decade: mean=2005.623, std=13.675


## 10. Save Cleaned Dataset

Export cleaned data to parquet format for efficient storage and fast loading.

In [12]:
# Select final columns for cleaned dataset
columns_to_keep = [
    'myanimelist_id', 'title', 'description', 'image', 'source_url',
    'Type', 'Episodes', 'Status', 'Premiered', 'Released_Season', 'Released_Year',
    'Source', 'Duration', 'Rating', 'Demographic',
    'Score', 'Ranked', 'Popularity', 'Members', 'Favorites',
    'Genres', 'genres_list', 'genres_count',
    'Themes', 'themes_list', 'themes_count',
    'Studios', 'studios_list', 'studios_count',
    'Producers', 'producers_list', 'producers_count',
    'character_list', 'character_count', 'character_names',
    'has_score', 'has_rank', 'is_completed',
    'log_members', 'log_favorites', 'favorite_rate',
    'primary_genre', 'primary_studio', 'is_modern', 'decade'
]

df_final = df_clean[columns_to_keep].copy()

# Save to parquet
output_path = PROCESSED_DIR / 'anime_clean.parquet'
df_final.to_parquet(output_path, index=False)

print(f"Cleaned dataset saved to: {output_path}")
print(f"\nFinal dataset shape: {df_final.shape}")
print(f"File size: {output_path.stat().st_size / (1024**2):.2f} MB")

print("\n" + "="*60)
print("CLEANING SUMMARY")
print("="*60)
print(f"Original shape: {df.shape}")
print(f"Final shape: {df_final.shape}")
print(f"Columns added: {df_final.shape[1] - df.shape[1]}")
print(f"\nKey statistics:")
print(f"  Anime with scores: {df_final['has_score'].sum()}")
print(f"  Anime with rankings: {df_final['has_rank'].sum()}")
print(f"  Completed anime: {df_final['is_completed'].sum()}")
print(f"  Modern anime (2010+): {df_final['is_modern'].sum()}")
print(f"  Unique genres: {df_final['primary_genre'].nunique()}")
print(f"  Unique studios: {df_final['primary_studio'].nunique()}")

Cleaned dataset saved to: data\processed\anime_clean.parquet

Final dataset shape: (19931, 45)
File size: 11.34 MB

CLEANING SUMMARY
Original shape: (19931, 25)
Final shape: (19931, 45)
Columns added: 20

Key statistics:
  Anime with scores: 15239
  Anime with rankings: 17406
  Completed anime: 18668
  Modern anime (2010+): 3868
  Unique genres: 22
  Unique studios: 1058


## 11. Verification

Load the saved parquet file to verify data integrity.

In [14]:
# Verify saved file
df_verify = pd.read_parquet(PROCESSED_DIR / 'anime_clean.parquet')

print("Verification successful")
print(f"Loaded shape: {df_verify.shape}")
print(f"Dtypes preserved: {(df_verify.dtypes == df_final.dtypes).all()}")

print("\nSample of cleaned data:")
print(df_verify[['title', 'Type', 'Score', 'primary_genre', 'genres_count', 'log_members']].head())

print("\n" + "="*60)
print("NOTEBOOK 01 COMPLETE")
print("="*60)


Verification successful
Loaded shape: (19931, 45)
Dtypes preserved: True

Sample of cleaned data:
                             title   Type  Score primary_genre  genres_count  \
0                     Cowboy Bebop     TV   8.75        Action             3   
1  Cowboy Bebop: Tengoku no Tobira  Movie   8.38        Action             2   
2                           Trigun     TV   8.22        Action             3   
3               Witch Hunter Robin     TV   7.23        Action             4   
4                   Bouken Ou Beet     TV   6.92        Action             3   

   log_members  
0    14.512660  
1    12.908192  
2    13.611116  
3    11.742997  
4     9.708506  

NOTEBOOK 01 COMPLETE
